# Modeling using PyGRF

In [2]:
!pip install PyGRF numpy pandas scikit-learn scipy
!pip install xarray rioxarray rasterio geopandas pyproj

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --------- ------------------------------ 2.9/12.6 MB 13.9 MB/s eta 0:00:01
   -------------------- ------------------- 6.3/12.6 MB 15.4 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.6 MB 14.6 MB/s eta 0:00:01
   ----------------------------------- ---- 11.3/12.6 MB 14.7 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.7 MB 15.2 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.7 MB 14.7 MB/s 


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from PyGRF import PyGRFBuilder
from pyproj import Transformer
from sklearn.impute import SimpleImputer

In [5]:
# load dataset
integrated = pd.read_csv('CIMIS_WBGT_all_2000_2025_integrated_fixed_2025_last8h.csv')

In [6]:
integrated.columns

Index(['station_id', 'year', 'month', 'day', 'hour', 'dday', 'lat', 'lon',
       'GEOID', 'solar', 'Tair', 'relhum', 'speed', 'elev_m', 'cza', 'fdir',
       'pres', 'zspeed', 'dT', 'urban', 'wbgt', 'aster_elev_mean',
       'aster_elev_sd', 'aster_elev_min', 'aster_elev_max', 'aster_elev_range',
       'aster_elev_median', 'aster_elev_q25', 'aster_elev_q75',
       'aster_elev_iqr', 'nlcd_urban', 'nlcd_imp_frac', 'nlcd_tree_frac',
       'nlcd_LC11_frac', 'nlcd_LC21_frac', 'nlcd_LC22_frac', 'nlcd_LC23_frac',
       'nlcd_LC24_frac', 'nlcd_LC31_frac', 'nlcd_LC41_frac', 'nlcd_LC42_frac',
       'nlcd_LC43_frac', 'nlcd_LC52_frac', 'nlcd_LC71_frac', 'nlcd_LC81_frac',
       'nlcd_LC82_frac', 'nlcd_LC90_frac', 'nlcd_LC95_frac',
       'nlcd_developed_frac', 'nlcd_forest_frac', 'nlcd_shrub_frac',
       'nlcd_water_frac', 'nlcd_crop_frac', 'nlcd_wetland_frac', 'modis_ndvi',
       'gridmet_rmax', 'gridmet_rmin', 'gridmet_sph', 'gridmet_srad',
       'gridmet_tmmn', 'gridmet_tmmx', 'gridmet

In [10]:
with pd.option_context('display.max_columns', None): display(integrated.head())

,station_id,year,month,day,hour,dday,lat,lon,GEOID,solar,Tair,relhum,speed,elev_m,cza,fdir,pres,zspeed,dT,urban,wbgt,aster_elev_mean,aster_elev_sd,aster_elev_min,aster_elev_max,aster_elev_range,aster_elev_median,aster_elev_q25,aster_elev_q75,aster_elev_iqr,nlcd_urban,nlcd_imp_frac,nlcd_tree_frac,nlcd_LC11_frac,nlcd_LC21_frac,nlcd_LC22_frac,nlcd_LC23_frac,nlcd_LC24_frac,nlcd_LC31_frac,nlcd_LC41_frac,nlcd_LC42_frac,nlcd_LC43_frac,nlcd_LC52_frac,nlcd_LC71_frac,nlcd_LC81_frac,nlcd_LC82_frac,nlcd_LC90_frac,nlcd_LC95_frac,nlcd_developed_frac,nlcd_forest_frac,nlcd_shrub_frac,nlcd_water_frac,nlcd_crop_frac,nlcd_wetland_frac,modis_ndvi,gridmet_rmax,gridmet_rmin,gridmet_sph,gridmet_srad,gridmet_tmmn,gridmet_tmmx,gridmet_vs,daymet_vp,daymet_tmin,daymet_tmax,daymet_srad,era5_wbgt
0,5,2000,1,1,0,1.000000,35.532556,-119.28179,144462,0,4.6,87,1.4,109.728,0.145924,0.000000e+00,1000.137467,10,-0.052,0,3.283895,106.555916,2.755774,93.0,116.0,23.0,107.0,105.0,108.0,3.0,0.0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.747215
1,5,2000,1,1,1,1.041667,35.532556,-119.28179,144462,1,3.8,88,1.1,109.728,0.031653,1.669446e-31,1000.137467,10,-0.052,0,2.474674,106.555916,2.755774,93.0,116.0,23.0,107.0,105.0,108.0,3.0,0.0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.666534
2,5,2000,1,1,2,1.083333,35.532556,-119.28179,144462,1,2.6,91,1.3,109.728,0.000000,0.000000e+00,1000.137467,10,-0.052,0,1.361490,106.555916,2.755774,93.0,116.0,23.0,107.0,105.0,108.0,3.0,0.0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.328669
3,5,2000,1,1,3,1.125000,35.532556,-119.28179,144462,1,1.8,92,1.2,109.728,0.000000,0.000000e+00,1000.137467,10,-0.052,0,0.559265,106.555916,2.755774,93.0,116.0,23.0,107.0,105.0,108.0,3.0,0.0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,3.766729
4,5,2000,1,1,4,1.166667,35.532556,-119.28179,144462,1,1.2,94,0.8,109.728,0.000000,0.000000e+00,1000.137467,10,-0.052,0,-0.097021,106.555916,2.755774,93.0,116.0,23.0,107.0,105.0,108.0,3.0,0.0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,3.504375


## preprocessing test dataset (month = Jan 2025)

In [21]:
# subset to January 2025 as the test month
df = integrated[
    (integrated["year"] == 2025) &
    (integrated["month"] == 1)
].copy()

print("Number of rows:", len(df))
print("Unique stations:", df["station_id"].nunique())
df.head()

Number of rows: 98120
Unique stations: 132


,station_id,year,month,day,hour,dday,lat,lon,GEOID,solar,...,gridmet_sph,gridmet_srad,gridmet_tmmn,gridmet_tmmx,gridmet_vs,daymet_vp,daymet_tmin,daymet_tmax,daymet_srad,era5_wbgt
21032404,2,2025,1,1,0,1.000000,36.336222,-120.11291,191939,0,...,0.003934,100.651985,274.4955,289.152,1.579834,729.44214,2.469104,16.402748,224.28404,4.320712
21032405,2,2025,1,1,1,1.041667,36.336222,-120.11291,191939,0,...,0.003934,100.651985,274.4955,289.152,1.579834,729.44214,2.469104,16.402748,224.28404,3.695096
21032406,2,2025,1,1,2,1.083333,36.336222,-120.11291,191939,0,...,0.003934,100.651985,274.4955,289.152,1.579834,729.44214,2.469104,16.402748,224.28404,2.643141
21032407,2,2025,1,1,3,1.125000,36.336222,-120.11291,191939,0,...,0.003934,100.651985,274.4955,289.152,1.579834,729.44214,2.469104,16.402748,224.28404,2.750543
21032408,2,2025,1,1,4,1.166667,36.336222,-120.11291,191939,0,...,0.003934,100.651985,274.4955,289.152,1.579834,729.44214,2.469104,16.402748,224.28404,3.594823


In [22]:
# Missingness within the selected month
print(
    df[["wbgt", "era5_wbgt", "modis_ndvi"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

# Station coverage
print(
    df.groupby("station_id")
    .size()
    .describe()
)

era5_wbgt     0.045485
modis_ndvi    0.003669
wbgt          0.000000
dtype: float64
count    132.000000
mean     743.333333
std        5.149174
min      702.000000
25%      744.000000
50%      744.000000
75%      744.000000
max      744.000000
dtype: float64


In [26]:
# Unique station locations
stations = df[
    ["station_id", "lat", "lon"]
].drop_duplicates("station_id")

print("Unique stations:", len(stations))
print("\nLatitude range:")
print(stations["lat"].agg(["min", "max"]))

print("\nLongitude range:")
print(stations["lon"].agg(["min", "max"]))

# Observations per station
coverage = df.groupby("station_id").size()

print("\nStations with fewer than 744 hours:",
      (coverage < 744).sum())

# Missingness of all predictors
print("\nMost incomplete predictors:")
print(
    df[features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

Unique stations: 132

Latitude range:
min    32.628208
max    41.958869
Name: lat, dtype: float64

Longitude range:
min   -124.243186
max   -114.558110
Name: lon, dtype: float64

Stations with fewer than 744 hours: 6

Most incomplete predictors:
era5_wbgt      0.045485
daymet_vp      0.007583
daymet_tmin    0.007583
daymet_tmax    0.007583
daymet_srad    0.007583
modis_ndvi     0.003669
solar          0.000000
relhum         0.000000
Tair           0.000000
zspeed         0.000000
dtype: float64


In [28]:
# --------------------------------------------------
# 1. remove rows missing target variable
# --------------------------------------------------

df = df.dropna(subset=["wbgt"]).copy()

# ============================================================
# 2. Define predictors using your actual column names
# ============================================================

features = [
    "era5_wbgt",

    # Hourly meteorology
    "solar", "Tair", "relhum", "speed",
    "elev_m", "cza", "fdir", "pres",
    "zspeed", "dT",
    
    "urban",

    # ASTER terrain statistics
    "aster_elev_mean", "aster_elev_sd",
    "aster_elev_min", "aster_elev_max",
    "aster_elev_range", "aster_elev_median",
    "aster_elev_q25", "aster_elev_q75",
    "aster_elev_iqr",

    # NLCD land-cover variables
    "nlcd_urban", "nlcd_imp_frac", "nlcd_tree_frac",
    "nlcd_LC11_frac", "nlcd_LC21_frac",
    "nlcd_LC22_frac", "nlcd_LC23_frac",
    "nlcd_LC24_frac", "nlcd_LC31_frac",
    "nlcd_LC41_frac", "nlcd_LC42_frac",
    "nlcd_LC43_frac", "nlcd_LC52_frac",
    "nlcd_LC71_frac", "nlcd_LC81_frac",
    "nlcd_LC82_frac", "nlcd_LC90_frac",
    "nlcd_LC95_frac", "nlcd_developed_frac",
    "nlcd_forest_frac", "nlcd_shrub_frac",
    "nlcd_water_frac", "nlcd_crop_frac",
    "nlcd_wetland_frac",

    # Vegetation
    "modis_ndvi",

    # GridMET
    "gridmet_rmax", "gridmet_rmin", "gridmet_sph",
    "gridmet_srad", "gridmet_tmmn", "gridmet_tmmx",
    "gridmet_vs",

    # Daymet
    "daymet_vp", "daymet_tmin", "daymet_tmax",
    "daymet_srad",
]

# Check whether every expected predictor exists.
missing_cols = [c for c in features if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing predictor columns: {missing_cols}")

# Convert predictors to numeric values.
df[features] = df[features].apply(
    pd.to_numeric, errors="coerce"
)

# ============================================================
# 3. Project longitude/latitude to metric coordinates
# ============================================================

# EPSG:3310 is California Albers; units are meters.
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3310", always_xy=True)

df["x"], df["y"] = transformer.transform(
    df["lon"].to_numpy(),
    df["lat"].to_numpy(),
)

coords = df[["x", "y"]].to_numpy()

# ============================================================
# 4. Create geographic groups for spatial validation
# ============================================================

# Use 20-km blocks as an initial example.
# Tune block size to the spatial extent and autocorrelation.
block_size = 20_000

df["block_x"] = np.floor(df["x"] / block_size).astype(int)
df["block_y"] = np.floor(df["y"] / block_size).astype(int)

df["spatial_block"] = (
    df["block_x"].astype(str) + "_"
    + df["block_y"].astype(str)
)

In [33]:
# ============================================================
# 5. Split into training and test spatial blocks
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["spatial_block"],
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# ============================================================
# 6. Impute missing predictor values
# ============================================================

# Fit the imputer on training data only.
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(train[features]), columns=features, index=train.index)
X_test = pd.DataFrame(imputer.transform(test[features]), columns=features, index=test.index)

y_train = train["wbgt"].to_numpy()
y_test = test["wbgt"].to_numpy()

coords_train = train[["x", "y"]].to_numpy()
coords_test = test[["x", "y"]].to_numpy()

## model with PyGRF

In [34]:
# ============================================================
# 7. Fit PyGRF
# ============================================================

# Start with a neighborhood of 100 observations.
# This is a starting value, not an optimized bandwidth.
model = PyGRFBuilder(
    n_estimators=200,
    max_features=1,
    band_width=100,
    train_weighted=True,
    predict_weighted=True,
    bootstrap=False,
    resampled=True,
    random_state=42,
)

model.fit(X_train, y_train, coords_train)

# Combine global and local predictions.
pred, pred_global, pred_local = model.predict(
    X_test,
    coords_test,
    local_weight=0.5,
)

# ============================================================
# 8. Evaluate predictions
# ============================================================

def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"Bias: {bias:.3f}")
    print(f"R2:   {r2:.3f}")


evaluate(y_test, pred, "PyGRF combined")
evaluate(y_test, pred_global, "PyGRF global")
evaluate(y_test, pred_local, "PyGRF local")

# Compare against the ERA5 WBGT baseline.
era5_test = test["era5_wbgt"].to_numpy()
valid = np.isfinite(era5_test)

if valid.any():
    evaluate(
        y_test[valid],
        era5_test[valid],
        "ERA5 WBGT baseline",
    )

# Save held-out predictions for further analysis.
results = test[
    ["station_id", "year", "month", "day", "hour",
     "lat", "lon", "wbgt", "era5_wbgt"]
].copy()

results["pred_pygrf"] = pred
results["pred_global"] = pred_global
results["pred_local"] = pred_local

results.to_csv("pygrf_test_predictions.csv", index=False)
print("\nSaved pygrf_test_predictions.csv")

MemoryError: Unable to allocate 46.3 GiB for an array with shape (78818, 78818) and data type float64

## try PyGRF with less data

In [38]:
# ============================================================
# 1. Reduce training data while retaining every station/day
# ============================================================

# Select one randomly chosen hourly observation per station per day.
train_small = (
    train.groupby(
        ["station_id", "day"],
        group_keys=False
    )
    .sample(n=1, random_state=42)
    .copy()
)

print("Original training rows:", len(train))
print("Sampled training rows:", len(train_small))
print("Training stations:", train_small["station_id"].nunique())

# ============================================================
# 2. Impute predictors using sampled training data only
# ============================================================

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(train_small[features]))
y_train = pd.Series(train_small["wbgt"].to_numpy())
coords_train = train_small[["x", "y"]].to_numpy()

# Prepare held-out test data.
X_test = pd.DataFrame(imputer.transform(test[features]))
y_test = pd.Series(test["wbgt"].to_numpy())
coords_test = test[["x", "y"]].to_numpy()

# ============================================================
# 3. Fit PyGRF
# ============================================================

model = PyGRFBuilder(
    n_estimators=200,
    max_features=1,
    band_width=100,
    train_weighted=True,
    predict_weighted=True,
    bootstrap=False,
    resampled=True,
    random_state=42,
)

model.fit(X_train, y_train, coords_train)

# ============================================================
# 4. Predict in batches to limit memory usage
# ============================================================

batch_size = 1000

pred = np.empty(len(test), dtype=float)
pred_global = np.empty(len(test), dtype=float)
pred_local = np.empty(len(test), dtype=float)

for start in range(0, len(test), batch_size):
    end = min(start + batch_size, len(test))

    result = model.predict(
        X_test[start:end],
        coords_test[start:end],
        local_weight=0.5,
    )

    pred[start:end] = result[0]
    pred_global[start:end] = result[1]
    pred_local[start:end] = result[2]

    print(f"Predicted {end:,} / {len(test):,} rows")

# ============================================================
# 5. Evaluate held-out station predictions
# ============================================================

def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"Bias: {bias:.3f}")
    print(f"R²:   {r2:.3f}")

evaluate(y_test, pred, "PyGRF combined")
evaluate(y_test, pred_global, "PyGRF global")
evaluate(y_test, pred_local, "PyGRF local")

# ERA5 baseline, evaluated on rows with valid ERA5 values.
era5 = test["era5_wbgt"].to_numpy()
valid = np.isfinite(era5)

if valid.any():
    evaluate(
        y_test[valid],
        era5[valid],
        "ERA5 WBGT baseline",
    )

# Save predictions.
results = test[
    [
        "station_id", "year", "month", "day", "hour",
        "lat", "lon", "wbgt", "era5_wbgt",
    ]
].copy()

results["pred_pygrf"] = pred
results["pred_global"] = pred_global
results["pred_local"] = pred_local

results.to_csv("pygrf_january2025_test.csv", index=False)
print("\nSaved pygrf_january2025_test.csv")

Original training rows: 78818
Sampled training rows: 3285
Training stations: 106
Predicted 1,000 / 19,302 rows
Predicted 2,000 / 19,302 rows
Predicted 3,000 / 19,302 rows


KeyboardInterrupt: 

In [39]:
from PyGRF import PyGRFBuilder
import time
import numpy as np

# Use the same sampled training data and imputed arrays.
# Start with a small test subset.
n_test = min(1000, len(X_test))

model_fast = PyGRFBuilder(
    n_estimators=50,       # Fewer trees for initial testing
    max_features=1,
    band_width=100,
    train_weighted=False,
    predict_weighted=False,
    bootstrap=False,
    resampled=False,
    random_state=42,
)

t0 = time.time()

model_fast.fit(
    X_train,
    y_train,
    coords_train,
)

print(f"Fit time: {(time.time() - t0) / 60:.2f} minutes")

t0 = time.time()

pred_fast = model_fast.predict(
    X_test[:n_test],
    coords_test[:n_test],
    local_weight=0.5,
)

print(f"Prediction time: {time.time() - t0:.1f} seconds")

KeyboardInterrupt: 

## compare random forest models

In [43]:
import numpy as np
import pandas as pd
import time

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ============================================================
# 1. Define the three models
# ============================================================

target = "wbgt"

# Model 1: ERA5 WBGT alone (baseline)
era5_feature = ["era5_wbgt"]

# Model 2: All predictors EXCEPT ERA5 WBGT
features_no_era5 = [
    f for f in features if f != "era5_wbgt"
]

# Model 3: All predictors INCLUDING ERA5 WBGT
features_with_era5 = features.copy()

# ============================================================
# 2. Use the same observations for a fair comparison
# ============================================================

# All models are evaluated on the same test rows.
# Require a valid target and coordinates.
test_eval = test.dropna(
    subset=["wbgt", "lat", "lon"]
).copy()

y_test = test_eval[target].to_numpy()

# ERA5 baseline can only be evaluated where ERA5 is available.
# Use this same subset for all three models.
valid_era5 = test_eval["era5_wbgt"].notna().to_numpy()

test_eval = test_eval.loc[valid_era5].copy()
y_test = test_eval[target].to_numpy()

print("Common test observations:", len(test_eval))
print("Held-out stations:", test_eval["station_id"].nunique())

# ============================================================
# 3. Define a reusable Random Forest function
# ============================================================

def fit_rf_and_predict(feature_cols, model_name):
    # Impute missing values using training data only.
    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(
        train_small[feature_cols]
    )

    X_test = imputer.transform(
        test_eval[feature_cols]
    )

    y_train = train_small[target].to_numpy()

    rf = RandomForestRegressor(
        n_estimators=100,
        max_features=1.0,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
    )

    start = time.time()
    rf.fit(X_train, y_train)
    fit_seconds = time.time() - start

    start = time.time()
    predictions = rf.predict(X_test)
    predict_seconds = time.time() - start

    print(
        f"{model_name}: "
        f"fit={fit_seconds:.1f}s, "
        f"predict={predict_seconds:.1f}s"
    )

    return predictions, rf

# ============================================================
# 4. Model 1: ERA5 baseline
# ============================================================

pred_era5 = test_eval["era5_wbgt"].to_numpy()

# ============================================================
# 5. Model 2: Random Forest without ERA5 WBGT
# ============================================================

pred_rf_no_era5, rf_no_era5 = fit_rf_and_predict(
    features_no_era5,
    "RF without ERA5 WBGT",
)

# ============================================================
# 6. Model 3: Random Forest with ERA5 WBGT
# ============================================================

pred_rf_with_era5, rf_with_era5 = fit_rf_and_predict(
    features_with_era5,
    "RF with ERA5 WBGT",
)

# ============================================================
# 7. Compare metrics
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAE": mean_absolute_error(
            y_true, y_pred
        ),
        "Bias": np.mean(y_pred - y_true),
        "R2": r2_score(y_true, y_pred),
    }

results = pd.DataFrame({
    "ERA5 baseline": calculate_metrics(
        y_test, pred_era5
    ),
    "RF without ERA5 WBGT": calculate_metrics(
        y_test, pred_rf_no_era5
    ),
    "RF with ERA5 WBGT": calculate_metrics(
        y_test, pred_rf_with_era5
    ),
}).T

print("\nModel comparison:")
print(results.round(4))

# ============================================================
# 8. Save metrics and predictions
# ============================================================

results.to_csv("wbgt_model_comparison_metrics.csv")

comparison = test_eval[
    [
        "station_id", "year", "month",
        "day", "hour", "lat", "lon", "wbgt",
        "era5_wbgt",
    ]
].copy()

comparison["pred_era5"] = pred_era5
comparison["pred_rf_no_era5"] = pred_rf_no_era5
comparison["pred_rf_with_era5"] = pred_rf_with_era5

comparison.to_csv(
    "wbgt_model_comparison_predictions.csv",
    index=False,
)

print("\nSaved model comparison metrics and predictions.")

Common test observations: 17814
Held-out stations: 24
RF without ERA5 WBGT: fit=3.4s, predict=0.2s
RF with ERA5 WBGT: fit=3.4s, predict=0.2s

Model comparison:
                        RMSE     MAE    Bias      R2
ERA5 baseline         2.8534  2.2469  0.3793  0.8273
RF without ERA5 WBGT  0.7604  0.4616 -0.0495  0.9877
RF with ERA5 WBGT     0.7948  0.4952 -0.0463  0.9866

Saved model comparison metrics and predictions.


## check correlation

In [42]:
# Correlations with WBGT in the training sample
corr = (
    train_small[features + ["wbgt"]]
    .corr(numeric_only=True)["wbgt"]
    .drop("wbgt")
    .sort_values(key=abs, ascending=False)
)

print(corr.head(20))

Tair                 0.914073
era5_wbgt            0.876371
solar                0.733950
gridmet_tmmn         0.432932
daymet_tmin          0.394868
aster_elev_min      -0.390420
aster_elev_q25      -0.388429
aster_elev_median   -0.386880
aster_elev_mean     -0.386795
aster_elev_q75      -0.385025
elev_m              -0.382855
aster_elev_max      -0.382385
pres                 0.382063
relhum              -0.348473
daymet_vp            0.336876
gridmet_tmmx         0.306200
daymet_tmax          0.295440
gridmet_sph          0.245779
speed                0.231592
fdir                 0.229711
Name: wbgt, dtype: float64


Fit three models using the exact same station split and test rows:

ERA5 baseline: era5_wbgt alone.

Random Forest without ERA5 WBGT: all other predictors.

Random Forest with ERA5 WBGT: your existing model.

Then compare RMSE, MAE, and bias for all three. This isolates the contribution of ERA5 WBGT and gives you a much stronger starting point for deciding whether PyGRF adds value.

Compare PyGRF, standard Random Forest, and the ERA5 WBGT baseline on the same held-out locations.

Evaluate whether results vary across California's coastal, Central Valley, mountain, and desert regions.